Selection:


In [ ]:
#### python
import os
import sys
import importlib
# columnar analysis
from coffea import processor
from coffea.nanoevents import NanoAODSchema
import awkward as ak
from dask.distributed import Client, performance_report
# local
sidm_path = str(os.getcwd()).split("/sidm")[0]
if sidm_path not in sys.path: sys.path.insert(1, sidm_path)
from sidm.tools import utilities, sidm_processor, scaleout, llpnanoaodschema
# always reload local modules to pick up changes during development
importlib.reload(utilities)
importlib.reload(sidm_processor)
importlib.reload(scaleout)
# plotting
import matplotlib.pyplot as plt
utilities.set_plot_style()
%matplotlib inline
import coffea.util
from matplotlib.colors import LogNorm
from coffea.processor import accumulate
import mplhep as hep
from helpers import merge_bkg, merge_cutflows

In [ ]:
vr = "31"
output = coffea.util.load(f"outputs/bkg_{vr}.coffea")

In [ ]:
# for k in output:
#     print(k)

In [ ]:
# Parameters
tmulxy1 = ["2Mu2E_500GeV_0p25GeV_0p004mm", "2Mu2E_500GeV_0p25GeV_0p04mm", "2Mu2E_500GeV_0p25GeV_0p4mm", "2Mu2E_500GeV_0p25GeV_2p0mm", "2Mu2E_500GeV_0p25GeV_4p0mm"]

#     "2Mu2E_500GeV_1p2GeV_0p019mm",  "2Mu2E_500GeV_1p2GeV_0p19mm",  "2Mu2E_500GeV_1p2GeV_1p9mm",  "2Mu2E_500GeV_1p2GeV_9p6mm",  "2Mu2E_500GeV_1p2GeV_19p0mm",
#     "2Mu2E_500GeV_5p0GeV_0p08mm",   "2Mu2E_500GeV_5p0GeV_0p08mm",  "2Mu2E_500GeV_5p0GeV_8p0mm",  "2Mu2E_500GeV_5p0GeV_40p0mm", "2Mu2E_500GeV_5p0GeV_80p0mm",
#     "2Mu2E_200GeV_1p2GeV_4p8mm",    "2Mu2E_500GeV_1p2GeV_1p9mm",   "2Mu2E_800GeV_1p2GeV_1p2mm",  "2Mu2E_1000GeV_1p2GeV_0p96mm",
# ]

fmulxy1 = ["4Mu_500GeV_0p25GeV_0p004mm",  "4Mu_500GeV_0p25GeV_0p04mm",   "4Mu_500GeV_0p25GeV_0p4mm",   "4Mu_500GeV_0p25GeV_2p0mm",   "4Mu_500GeV_0p25GeV_4p0mm"]

#     "4Mu_500GeV_1p2GeV_0p019mm",   "4Mu_500GeV_1p2GeV_0p19mm",    "4Mu_500GeV_1p2GeV_1p9mm",    "4Mu_500GeV_1p2GeV_9p6mm",    "4Mu_500GeV_1p2GeV_19p0mm", 
#     "4Mu_500GeV_5p0GeV_0p08mm",    "4Mu_500GeV_5p0GeV_0p8mm",     "4Mu_500GeV_5p0GeV_8p0mm",    "4Mu_500GeV_5p0GeV_40p0mm",   "4Mu_500GeV_5p0GeV_80p0mm",
#     "4Mu_200GeV_1p2GeV_4p8mm",     "4Mu_500GeV_1p2GeV_1p9mm",     "4Mu_800GeV_1p2GeV_1p2mm",    "4Mu_1000GeV_1p2GeV_0p96mm",
# ]

bkgttj = ["TTJets"]

bkgdyj = ["DYJetsToMuMu_M10to50", "DYJetsToMuMu_M50"]

bkgqcd = ["QCD_Pt15To20", "QCD_Pt20To30", "QCD_Pt30To50", "QCD_Pt50To80", "QCD_Pt80To120", "QCD_Pt120To170", "QCD_Pt170To300", "QCD_Pt300To470", 
          "QCD_Pt470To600", "QCD_Pt600To800", "QCD_Pt800To1000", "QCD_Pt1000"]

channels = ["baseNoLj",
            "bkg_base", 
            "bkg_base_iso",
            "bkg_base_iso_disp",
            "bkg_base_iso_disp_2mu2e",
            "bkg_base_iso_disp_2mu2e_dphi",
            "bkg_base_iso_disp_4mu",
            "bkg_base_iso_disp_4mu_dphi",
]

ch1 = channels[0]
ch2 = channels[1]
ch3 = channels[2]
ch4 = channels[3]
ch5 = channels[4]
ch6 = channels[5]
ch7 = channels[6]
ch8 = channels[7]


cha2name = ["Base", "Iso", "Disp", "2mu2e", r"$\Delta\Phi > 2$"]
cha2 = [ch2, ch3, ch4, ch5, ch6,]

cha4name = ["Base", "Iso", "Disp", "4mu", r"$\Delta\Phi > 2$"]
cha4 = [ch2, ch3, ch4, ch7, ch8,]

histoplot = ["lj_lj_invmass"]

lablxy1 = ["0p3", "3p0", "30", "150", "300"]
colsig=["black", "darkviolet", "navy", "magenta", "darkorange", "darkblue"]
# labmx=[r"$2\mu2e: m_{xx}=200$", r"$2\mu2e: m_{xx}=500$", r"$2\mu2e: m_{xx}=800$", r"$2\mu2e: m_{xx}=1000$"]

figh, figw = 12, 10


# Starting with the 2mu channel

## Looking at how each selection affect each signal and background process separately

In [ ]:
#2mu signal + bkg sep
for ik, htp in enumerate(histoplot):
    plt.subplots(1, 1, figsize=(figh, figw))
    for ij, ch in enumerate(cha2):
        utilities.plot(output[tmulxy1[2]]["hists"][htp][ch, :], label = cha2name[ij])
    plt.title("Signal")
    plt.ylabel("Events")
    plt.yscale("log")
    plt.legend(title=r"Selection:", alignment="left", loc=0)
    plt.text(0.05, 0.95,
        "m$_{XX}$ = 500 GeV\nm$_{Z_D}$ = 0.25 GeV\n$l_{xy}$ = 30cm",
        transform=plt.gca().transAxes,
        fontsize=24,
        va="top",)

#TTJets
for ik, htp in enumerate(histoplot):
    plt.subplots(1, 1, figsize=(figh, figw))
    for ij, ch in enumerate(cha2):
        TTJ = merge_bkg(output, bkgttj, htp, ch)
        utilities.plot(TTJ, label = cha2name[ij])
    plt.title("TTJets")
    plt.ylabel("Events")
    plt.yscale("log")
    plt.legend(title=r"Selection:", alignment="left", loc=0)

#DYJets
for ik, htp in enumerate(histoplot):
    plt.subplots(1, 1, figsize=(figh, figw))
    for ij, ch in enumerate(cha2):
        DYJ = merge_bkg(output, bkgdyj, htp, ch)
        utilities.plot(DYJ, label = cha2name[ij])
    plt.title("DYJets")
    plt.ylabel("Events")
    plt.yscale("log")
    plt.legend(title=r"Selection:", alignment="left", loc=0)

#DYJets
for ik, htp in enumerate(histoplot):
    plt.subplots(1, 1, figsize=(figh, figw))
    for ij, ch in enumerate(cha2):
        QCD = merge_bkg(output, bkgqcd, htp, ch)
        utilities.plot(QCD, label = cha2name[ij])
    plt.title("QCD")
    plt.ylabel("Events")
    plt.yscale("log")
    plt.legend(title=r"Selection:", alignment="left", loc=0)

## Stacking Backgrounds for different selections for $m_{xx} = 500$, $m_{z_D} = 0.25$ and changing $l_{xy}$

In [ ]:
#4mu signal
for ik, htp in enumerate(histoplot):
    plt.subplots(1, 1, figsize=(figh, figw))
    for ij, ch in enumerate(cha4):
        utilities.plot(output[fmulxy1[2]]["hists"][htp][ch, :], label = cha4name[ij])
    plt.title("Signal")
    plt.ylabel("Events")
    plt.yscale("log")
    plt.legend(title=r"Selection:", alignment="left", loc=0)
    plt.text(0.05, 0.95,
        "m$_{XX}$ = 500 GeV\nm$_{Z_D}$ = 0.25 GeV\n$l_{xy}$ = 30cm",
        transform=plt.gca().transAxes,
        fontsize=24,
        va="top",)

#TTJets
for ik, htp in enumerate(histoplot):
    plt.subplots(1, 1, figsize=(figh, figw))
    for ij, ch in enumerate(cha4):
        TTJ = merge_bkg(output, bkgttj, htp, ch)
        utilities.plot(TTJ, label = cha4name[ij])
    plt.title("TTJets")
    plt.ylabel("Events")
    plt.yscale("log")
    plt.legend(title=r"Selection:", alignment="left", loc=0)

#DYJets
for ik, htp in enumerate(histoplot):
    plt.subplots(1, 1, figsize=(figh, figw))
    for ij, ch in enumerate(cha4):
        DYJ = merge_bkg(output, bkgdyj, htp, ch)
        utilities.plot(DYJ, label = cha2name[ij])
    plt.title("DYJets")
    plt.ylabel("Events")
    plt.yscale("log")
    plt.legend(title=r"Selection:", alignment="left", loc=0)

#DYJets
for ik, htp in enumerate(histoplot):
    plt.subplots(1, 1, figsize=(figh, figw))
    for ij, ch in enumerate(cha4):
        QCD = merge_bkg(output, bkgqcd, htp, ch)
        utilities.plot(QCD, label = cha2name[ij])
    plt.title("QCD")
    plt.ylabel("Events")
    plt.yscale("log")
    plt.legend(title=r"Selection:", alignment="left", loc=0)

## Stacking Backgrounds for different selections for $m_{xx} = 500$, $m_{z_D} = 0.25$ and changing $l_{xy}$

In [ ]:
# 4mu
###################################################
####### Stacking Backgrounds diff selections
for htp in histoplot:  
    for ic, ch in enumerate(cha2):
        
        fig, ax = plt.subplots(figsize=(figh, figw))
        
        DYJ = merge_bkg(output, bkgdyj, htp, ch)
        QCD = merge_bkg(output, bkgqcd, htp, ch)
        TTJ = merge_bkg(output, bkgttj, htp, ch)
        
        # Backgrounds
        hep.histplot([DYJ, QCD, TTJ],
                     stack=True,  histtype="fill", 
                     color=["#FFD700", "#3498DB", "#E74C3C"],
                     label=["DYJ", "QCD", "TTJ"],
                     ax=ax, )
        
        # Signal        
        for ix, sss in enumerate(tmulxy1):
            hep.histplot(output[sss]["hists"][htp][ch1, ::2j],  color=colsig[ix], label=lablxy1[ix], histtype="step", linewidth=3, ax=ax)
        hep.cms.label(data=False, lumi=None, com=13, ax=ax)
        # ax.set_ylim(1, 1000000)
        # ax.yscale(1,10000)
        ax.set_yscale("log")
        ax.legend(title="Process", ncol=2)
        plt.title(f"selection: {cha2name[ic]}")
        # plt.savefig(f"clean_plots/fin_bkg_stack_{vr}_{ht}_{cha2name[ic]}_mxx.pdf", bbox_inches="tight", dpi=300)

## More LJ variables for background and signal for final cuts

In [ ]:
mhtoplot = ["lj_pt", "lj_pfMuon_pt"]
mhtoname = [r"Lepton-jet $p_T$", "Lepton-jet muon $p_T$"]
proname = [r"$4\mu"]
npl = 2

for ih, htp in enumerate(mhtoplot):
    plt.subplots(1, 2, figsize=(npl*figh, figw))
    plt.subplot(1,2,1)
    utilities.plot(output[tmulxy1[2]]["hists"][htp][ch6, ::2j], label = r"$2\mu2e$", color = "black")
    
    DYJ = merge_bkg(output, bkgdyj, htp, ch6)
    utilities.plot(DYJ, label="DYJ", color="#FFD700")
    
    TTJ = merge_bkg(output, bkgttj, htp, ch6)
    utilities.plot(TTJ, label="TTJ", color="#E74C3C")

    QCD = merge_bkg(output, bkgqcd, htp, ch6)
    utilities.plot(QCD, label="QCD", color="#3498DB")
    
    plt.yscale("log")
    plt.legend(title="Selections", alignment="left", loc=0)
    plt.title(mhtoname[ih])

    
    plt.subplot(1,2,2)
    utilities.plot(output[fmulxy1[2]]["hists"][htp][ch8, ::2j], label = r"$4\mu$", color = "black")

    DYJ = merge_bkg(output, bkgdyj, htp, ch8)
    utilities.plot(DYJ, label="DYJ", color="#FFD700")
    
    TTJ = merge_bkg(output, bkgttj, htp, ch8)
    utilities.plot(TTJ, label="TTJ", color="#E74C3C")

    QCD = merge_bkg(output, bkgqcd, htp, ch8)
    utilities.plot(QCD, label="QCD", color="#3498DB")
    
    plt.yscale("log")
    plt.legend(title="Selections", alignment="left", loc=0)
    plt.title(mhtoname[ih])

## Final yield summary

In [ ]:
chan = ch8        # Change this once
dphi = "LJ-LJ dPhi > 2"

print(f"{'Sample':<30} {'Yield':>12}")
print("-" * 44)

# Signal yields
for sample in fmulxy1:
    print(f"{sample:<30} {output[sample]['cutflow'][chan].rows[dphi]['weighted']:>12.2f}")
    print("-" * 44)

# Background yields
dyj = merge_cutflows(output, bkgdyj, chan)
ttj = merge_cutflows(output, bkgttj, chan)
qcd = merge_cutflows(output, bkgqcd, chan)

print(f"{'TTJets':<30} {ttj[dphi]['weighted']:>12.2f}")
print("-" * 44)
print(f"{'DYJ':<30}    {dyj[dphi]['weighted']:>12.2f}")
print("-" * 44)
print(f"{'QCD':<30}    {qcd[dphi]['weighted']:>12.2f}")

## Number of events after all cuts have been applied

In [ ]:
for i in tmulxy1:
    print(i)
    output[i]["cutflow"][ch6].print_table()
    print()

In [ ]:
DYJ = merge_cutflows(output, bkgdyj, ch6)
TTJ = merge_cutflows(output, bkgttj, ch6)
QCD = merge_cutflows(output, bkgqcd, ch6)

backgrounds = {
    "DYJ": bkgdyj,
    "TTJ": bkgttj,
    "QCD": bkgqcd,
}

merged_cf = {}

for name, samples in backgrounds.items():
    merged_cf[name] = merge_cutflows(output, samples, ch6)

for name, cf in merged_cf.items():
    print(f"\n{name}")
    print(f"{'cut name':<20} {'raw':>12} {'weighted':>15}")
    print("-" * 50)

    for cut, vals in cf.items():
        print(
            f"{cut:<20}"
            f"{vals['raw']:>12,.0f}"
            f"{vals['weighted']:>15,.1f}"
        )

In [ ]:
all_bkgs = bkgdyj + bkgttj + bkgqcd
total_bkg = merge_cutflows(output, all_bkgs, ch6)

for cut, vals in total_bkg.items():
    print(
        f"{cut:<20}"
        f"{vals['raw']:>12,.0f}"
        f"{vals['weighted']:>15,.1f}"
    )

In [ ]:
vr = "31"
output = coffea.util.load(f"outputs/bkg_{vr}.coffea")
cf = output["2Mu2E_500GeV_0p25GeV_0p004mm"]["cutflow"][ch6]
cf.print_table()

In [ ]:
# check values for one sample
sample = "2Mu2E_500GeV_0p25GeV_0p004mm"
print(output[sample].keys())
print(output[sample]["cutflow"][ch6].rows["None"])
print(output[sample]["cutflow"][ch6].rows["LJ-LJ dPhi > 2"])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

ctau = np.array([0.004, 0.04, 0.4, 2.0, 4.0])
def get_acc_eff(output, sample, channel, final_cut="LJ-LJ dPhi > 2"):
    cf = output[sample]["cutflow"][channel].rows

    n_gen = cf["None"]["raw"]
    n_sel = cf[final_cut]["raw"]

    return n_sel / n_gen


acc_eff = []

for s in tmulxy1:
    acc_eff.append(get_acc_eff(output, s, ch6))

acc_eff = np.array(acc_eff)

fig, ax = plt.subplots(figsize=(8,6))

ax.plot(ctau, acc_eff, marker='o', linewidth=2)

ax.set_xscale("log")
ax.set_xlabel("cτ [mm]")
ax.set_ylabel("Acceptance × Efficiency")
ax.set_title("Signal Acceptance × Efficiency vs cτ")

ax.grid(True, which="both", linestyle=":")

plt.tight_layout()
plt.show()  